# Урок 24 — Структури Даних: Список як Архітектурне Рішення

**Модуль 3 · Python Advanced · Автор: Viktor Nikoriak**

---

## Про що цей урок?

Більшість розробників знають `list`. Але senior-інженер розуміє, що
«список» — це не конкретна структура, а **абстракція**, яку можна реалізувати
двома принципово різними способами:

| Спосіб | Фізична модель | Головний виграш | Головна жертва |
|--------|----------------|-----------------|----------------|
| **Динамічний масив** (`list`) | Суцільний блок RAM | O(1) довільний доступ | O(n) вставка/видалення |
| **Зв'язний список** (Linked List) | Вузли-острівці по всій RAM | O(1) вставка/видалення | O(n) пошук за індексом |

Цей урок — про те, **чому це важливо** і як правильно будувати власні структури даних.

| Частина | Тема |
|---------|------|
| 1 | Python `list` як динамічний масив — архітектура пам'яті |
| 📜 | Екскурс в Історію — від Зусе (1945) до Сазерленда (1963) |
| 2 | Зв'язний список — вузли, голова, хвіст |
| 3 | Реалізація `Node` + `LinkedList` у Python |
| 4 | ADT: інтерфейс vs реалізація — чому ми ховаємо деталі |
| 5 | Паттерн Ітератор — `__iter__`, `__next__`, `yield` |
| 6 | Операції: обхід, вставка, видалення, побудова |
| 7 | Стек і Черга на вузлах — Node-based реалізація |
| 8 | Двосторонній зв'язний список + `collections.deque` |
| 9 | Паттерни ООП: Адаптер, Відвідувач, Компонент |
| 10 | Питання для співбесіди + Домашні завдання |

---

---

## Частина 1: Python `list` — Динамічний Масив

---

### Архітектура пам'яті

Python `list` — це **не зв'язний список**. Під капотом це **динамічний масив** —
єдиний суцільний блок RAM, що містить масив *вказівників* (посилань) на реальні об'єкти.

```
Пам'ять:  ┌──────┬──────┬──────┬──────┬──────┐
           │ ptr0 │ ptr1 │ ptr2 │ ptr3 │ ptr4 │   ← масив вказівників
           └──┬───┴──┬───┴──┬───┴──┬───┴──┬───┘
              │      │      │      │      │
             42    "hi"   True   [...]  None        ← реальні об'єкти
```

**Чому `list[i]` — O(1)?**
Адреса елемента: `base_address + i × pointer_size`. Обчислення за одну дію — незалежно від розміру списку.

**Чому `list.insert(0, x)` — O(n)?**
Щоб вставити на початок, процесор мусить фізично зсунути **всі** n вказівників вправо.
При мільйоні елементів — мільйон операцій зсуву.

In [ ]:
import sys
import timeit

# Демонстрація O(1) random access vs O(n) insert at beginning

sizes = [1_000, 10_000, 100_000]

print(f"{'Розмір':>10} | {'list[i] (мкс)':>15} | {'insert(0,x) (мкс)':>20}")
print("-" * 55)

for n in sizes:
    t_access = timeit.timeit(
        stmt='lst[n // 2]',
        setup=f'lst = list(range({n})); n = {n}',
        number=10_000
    ) / 10_000 * 1e6  # мікросекунди

    t_insert = timeit.timeit(
        stmt='lst.insert(0, -1); lst.pop(0)',  # вставка + відновлення
        setup=f'lst = list(range({n}))',
        number=1_000
    ) / 1_000 * 1e6

    print(f"{n:>10,} | {t_access:>15.2f} | {t_insert:>20.2f}")

print()
print("Висновок:")
print("  list[i]      → O(1): час практично не змінюється зі зростанням n")
print("  insert(0, x) → O(n): час зростає лінійно разом із розміром")

---

## Екскурс в Історію: Як З'явився Динамічний Масив

---

### Передісторія (1940-ві роки)

Перші комп'ютери вже мали справу зі списками даних. **Конрад Зусе (Konrad Zuse)**,
німецький інженер, у **1945 році** створив перші нетривіальні алгоритми для роботи
зі списками, довжина яких динамічно змінюється — фактично заклавши основу всього,
що ми розглядаємо в цьому уроці.

### Народження сучасного динамічного масиву (1960-ті)

Концепція суцільного масиву змінної довжини, де елементи займають послідовні
адреси пам'яті і можуть зсуватися або розширюватися, виникла незалежно у кількох місцях:

| Рік | Хто | Де | Що |
|-----|-----|----|----|
| **1963** | **Дж. Данлеп (J. Dunlap)** | Корпорація Digitek | Розробив технологію під час написання компіляторів |
| **~1963** | **IBM Corporation** | Компілятор COBOL + підсистема CITRUS | Незалежно розробили ту саму ідею |
| **1964** | **Ян Гарвік (Jan Garwick)** | Норвегія | Перша **публікація** концепції — до нього всі розробки були комерційними таємницями |

### Принцип Over-Allocation (надлишкове виділення)

Ключова ідея динамічного масиву:

1. При створенні виділяється **більше** пам'яті, ніж потрібно зараз
2. Коли масив заповнюється — виділяється новий блок (зазвичай вдвічі більший)
3. Всі елементи **копіюються** у новий блок
4. Старий блок звільняється

Саме цей принцип лежить в основі Python `list`.
Стандарт **ISO C99 (1999)** офіційно ввів масиви змінної довжини на рівні мови C,
позбавивши розробників від ручного `malloc/realloc`.

In [ ]:
import sys

# ── Демонстрація Over-Allocation у Python list ─────────────────────────────
# sys.getsizeof() повертає реальний розмір об'єкта в байтах (включно з буфером)

print("Демонстрація over-allocation: Python list росте 'стрибками'")
print()
print(f"{'Кількість елементів':>22} | {'getsizeof (байт)':>18} | {'Відмінність':>12}")
print("-" * 60)

prev_size = 0
lst = []
for i in range(33):
    size = sys.getsizeof(lst)
    diff = f"+{size - prev_size}" if size != prev_size else "—"
    print(f"{i:>22} | {size:>18} | {diff:>12}")
    prev_size = size
    lst.append(i)

print()
print("Спостереження: розмір зростає не на кожному append,")
print("а 'стрибками' — це і є over-allocation в дії.")
print("CPython використовує формулу: new_capacity = (old + 3) | 3  (наближення до потужностей 2)")

### Народження Зв'язного Списку (1946–1963)

Ідея зберігати елементи у **нескладних клітинах пам'яті з вказівниками** виникла
з апаратних особливостей перших комп'ютерів із **магнітними барабанами**.

Для оптимізації швидкості використовувалася «адресація 1+1» — кожна інструкція
містила адресу **наступної** інструкції. Це і є зародкова концепція вказівника.

| Рік | Хто | Внесок |
|-----|-----|--------|
| **1946** | **Джон Моклі (John Mauchly)** | Описав «адресацію 1+1» — зародок ідеї зв'язного списку |
| **1953** | **Г. П. Лан (H. P. Luhn)** | Запропонував «ланцюжки» для алгоритмів зовнішнього пошуку |
| **1956** | **Ньюелл, Шоу, Саймон** (Newell, Shaw, Simon) | Розробили **IPL-II** — першу мову обробки списків, засновану на вказівниках. Їхній підхід настільки вплинув на спільноту, що цей тип пам'яті назвали **NSS-пам'яттю** |
| **1959** | **Дж. В. Карр III (J. W. Carr III)** | Довів, що зв'язними списками можна ефективно маніпулювати у звичайних мовах програмування |
| **1963** | **Айвен Сазерленд (Ivan Sutherland)** | Запровадив **двоспрямовані зв'язні списки** (Doubly Linked Lists) у своїй революційній графічній системі **Sketchpad** |

> **Висновок:** `collections.deque` у Python 2024 року — прямий нащадок
> двоспрямованого зв'язного списку Сазерленда з 1963 року.
> Фундаментальні ідеї не старіють — вони лише отримують нові реалізації.

---

---

## Частина 2: Зв'язний Список — Децентралізована Архітектура

---

### Ментальна модель: відмова від монолітності

Зв'язний список **не вимагає суцільного блоку RAM**. Дані зберігаються
в незалежних об'єктах — **вузлах (Nodes)** — які можуть бути розкидані
у будь-яких вільних куточках пам'яті.

```
RAM (фрагментована):

  адр. 0x1A:  ┌─────────┬──────────┐
               │ data=10 │ next→0x4F│    Вузол 1 (Head)
               └─────────┴──────────┘

  адр. 0x4F:  ┌─────────┬──────────┐
               │ data=20 │ next→0xB2│    Вузол 2
               └─────────┴──────────┘

  адр. 0xB2:  ┌─────────┬──────────┐
               │ data=30 │ next=None│    Вузол 3 (Tail)
               └─────────┴──────────┘
```

### Складність операцій

| Операція | Зв'язний список | Python `list` |
|----------|-----------------|---------------|
| Доступ за індексом `[i]` | **O(n)** ← треба йти від head | **O(1)** |
| Вставка/видалення на початку | **O(1)** ← змінюємо вказівник | **O(n)** ← зсуваємо масив |
| Вставка/видалення в середині | **O(1)** (якщо є вказівник на вузол) | **O(n)** |
| Пошук значення | **O(n)** | **O(n)** |
| Пам'ять на елемент | Більше (+ вказівник `next`) | Менше |

> **Архітектурна мудрість:** Жодна структура не є "кращою" в абсолюті.
> Вибір залежить від того, яка операція — домінуюча у вашому алгоритмі.

---

## Частина 3: Реалізація Node + LinkedList у Python

---

### Клас `Node` — базовий будівельний блок

Python не має вбудованого зв'язного списку — ми будуємо його через ООП.
Перший крок: клас `Node`, що інкапсулює **дані + вказівник**.

In [ ]:
# ────────────────────────────────────────────────────────
# Базовий будівельний блок зв'язного списку
# ────────────────────────────────────────────────────────

class ListNode:
    """Один вузол однонаправленого зв'язного списку."""
    
    def __init__(self, data=None, next_node=None):
        self.data = data        # корисне навантаження — будь-який Python об'єкт
        self.next = next_node   # вказівник на наступний вузол (або None → кінець)
    
    def __repr__(self):
        return f"Node({self.data!r})"


# Ручне з'єднання вузлів для демонстрації
node3 = ListNode(30)
node2 = ListNode(20, next_node=node3)
node1 = ListNode(10, next_node=node2)  # head

print("Структура ланцюжка:")
current = node1
while current is not None:
    arrow = " → " if current.next else " → None"
    print(f"  {current}{arrow}", end="")
    current = current.next
print()

print()
print(f"node1.data = {node1.data}")
print(f"node1.next = {node1.next}")
print(f"node1.next.next = {node1.next.next}")

### Клас `LinkedList` — обгортка над вузлами

Клієнтський код **ніколи** не повинен взаємодіяти з `Node` напряму.
Клас `LinkedList` приховує внутрішню "сантехніку" і надає чистий API.

In [ ]:
class LinkedList:
    """Однонаправлений зв'язний список з інкапсульованою архітектурою."""
    
    def __init__(self):
        self._head = None   # вказівник на перший вузол (прихований)
        self._size = 0      # кешований розмір — щоб len() був O(1)
    
    # ── Вставка ──────────────────────────────────────────────
    
    def prepend(self, data) -> None:
        """Вставляє елемент на початок — O(1)."""
        # Новий вузол → вказує на поточний head → стає новим head
        self._head = ListNode(data, next_node=self._head)
        self._size += 1
    
    def append(self, data) -> None:
        """Вставляє елемент у кінець — O(n): треба дійти до tail."""
        new_node = ListNode(data)
        if self._head is None:          # порожній список
            self._head = new_node
        else:
            current = self._head
            while current.next:         # доходимо до останнього вузла
                current = current.next
            current.next = new_node     # чіпляємо новий вузол у кінець
        self._size += 1
    
    # ── Пошук ────────────────────────────────────────────────
    
    def search(self, key) -> bool:
        """Перевіряє наявність значення — O(n)."""
        current = self._head
        while current:
            if current.data == key:
                return True
            current = current.next
        return False
    
    # ── Видалення ─────────────────────────────────────────────
    
    def remove(self, key) -> bool:
        """Видаляє перший вузол з вказаним значенням — O(n)."""
        if self._head is None:
            return False
        
        # Особливий випадок: видаляємо голову
        if self._head.data == key:
            self._head = self._head.next
            self._size -= 1
            return True
        
        # Загальний випадок: знаходимо попередній вузол
        prev = self._head
        while prev.next:
            if prev.next.data == key:
                prev.next = prev.next.next   # обхідний шлях
                self._size -= 1
                return True
            prev = prev.next
        return False
    
    # ── Python-протоколи ──────────────────────────────────────
    
    def __len__(self) -> int:
        return self._size
    
    def __repr__(self) -> str:
        parts, current = [], self._head
        while current:
            parts.append(str(current.data))
            current = current.next
        return " → ".join(parts) + " → None" if parts else "Empty"


# ── Демонстрація ──────────────────────────────────────────────────────────
ll = LinkedList()

# Додаємо елементи
for val in [10, 20, 30]:
    ll.append(val)
print(f"Після append 10,20,30 : {ll}")

ll.prepend(5)
print(f"Після prepend(5)      : {ll}  | len={len(ll)}")

# Пошук
print(f"search(20) → {ll.search(20)}")
print(f"search(99) → {ll.search(99)}")

# Видалення
ll.remove(20)
print(f"Після remove(20)      : {ll}  | len={len(ll)}")

ll.remove(5)   # видаляємо голову
print(f"Після remove(5/head)  : {ll}  | len={len(ll)}")

---

## Частина 4: ADT та Інкапсуляція — Контракт проти Секрету

---

### Чому ми ховаємо деталі реалізації?

Це **не** про безпеку від хакерів. Це про **фізичну незалежність даних**.

> Якщо ви відкриєте `Node` та `next`-вказівники назовні, клієнти напишуть:
> `while current_node.next is not None: ...`
> Після цього ви **назавжди** прив'язані до зв'язного списку.
> Змінити реалізацію на масив — зламає весь клієнтський код.

**Ховаючи внутрішнє, ви залишаєте собі свободу:**
переписати структуру даних з нуля, не зламавши жодного рядка у клієнтів.

### Python-конвенції приховування

| Конвенція | Приклад | Значення |
|-----------|---------|----------|
| Одне підкреслення | `self._head` | "Внутрішня деталь — не чіпай без потреби" |
| Два підкреслення | `self.__data` | Name mangling — Python перейменовує атрибут |
| Dunder-методи | `__len__`, `__iter__` | Публічний інтерфейс через Python-протоколи |

### `collections.abc` — стандартні ABC для структур даних

```python
from collections.abc import Iterable, Sized, Container

# Якщо клас наслідує Sized — МУСИТЬ реалізувати __len__
# Якщо не реалізував — TypeError при instantiation
```

| ABC | Потрібні методи | Що дає |
|-----|-----------------|--------|
| `Sized` | `__len__` | `len(obj)` |
| `Iterable` | `__iter__` | `for x in obj:` |
| `Container` | `__contains__` | `x in obj` |
| `Sequence` | `__getitem__`, `__len__` | Індексація + ітерація |

In [ ]:
from collections.abc import Sized, Iterable, Container


class StrictLinkedList(Sized, Iterable, Container):
    """
    Зв'язний список, що явно заявляє свої ABC-контракти.
    Python перевірить: якщо __len__, __iter__ або __contains__ не реалізовані —
    TypeError при спробі створення екземпляра.
    """
    
    def __init__(self):
        self._head = None
        self._size = 0
    
    def add(self, data) -> None:
        """Публічний API — клієнт не знає нічого про Node."""
        self._head = ListNode(data, next_node=self._head)
        self._size += 1
    
    # ── ABC-контракти (публічний інтерфейс) ──────────────────
    
    def __len__(self) -> int:           # Sized
        return self._size
    
    def __contains__(self, item) -> bool:  # Container
        current = self._head
        while current:
            if current.data == item:
                return True
            current = current.next
        return False
    
    def __iter__(self):                 # Iterable (через generator)
        current = self._head
        while current:
            yield current.data          # клієнт бачить тільки дані, не вузли
            current = current.next
    
    def __repr__(self) -> str:
        return f"StrictLinkedList([{', '.join(str(x) for x in self)}])"


# ── Демонстрація контрактів ────────────────────────────────────────────────
sll = StrictLinkedList()
for val in [10, 20, 30, 40]:
    sll.add(val)

# Клієнт працює через стандартний Python-інтерфейс:
print(f"len(sll)      = {len(sll)}")          # Sized   → __len__
print(f"30 in sll     = {30 in sll}")          # Container → __contains__
print(f"99 in sll     = {99 in sll}")

print("for x in sll  :", end=" ")             # Iterable → __iter__
for x in sll:
    print(x, end=" ")
print()

print(f"repr          = {sll}")

# Клієнт НІЧОГО не знає про ListNode або _head

---

## Частина 5: Паттерн Ітератор — `yield` як Суперсила

---

### Проблема: як обійти структуру, не розкриваючи вузли?

Якщо масив — просто `for i in range(len(arr))`,
то зв'язний список потребує `current = current.next`.
Клієнт **не повинен** знати, яка механіка всередині.

### Рішення: Паттерн Ітератор

Структура даних надає **ітераторний об'єкт**, який:
1. Зберігає поточний стан обходу (`current = node`)
2. Видає клієнту лише **дані** через `__next__()`
3. Сигналізує кінець через `StopIteration`

### Pythonic рішення: `yield` (генератор)

Замість окремого класу-ітератора, Python дозволяє використати
**генераторну функцію** прямо в методі `__iter__`.
Python автоматично будує ітераторний об'єкт і керує станом.

In [ ]:
# ── Демонстрація: як yield "заморожує" виконання ──────────────────────────

def countdown_generator(n: int):
    """Генераторна функція — демонстрація механізму yield."""
    print(f"  [gen] Старт: починаємо від {n}")
    while n > 0:
        print(f"  [gen] Перед yield {n}")
        yield n           # ПАУЗА: повертаємо значення, зберігаємо стан
        print(f"  [gen] Після yield {n}")  # відновлюємось тут при наступному next()
        n -= 1
    print("  [gen] StopIteration")


print("Ітеруємо countdown_generator(3):")
gen = countdown_generator(3)
for val in gen:
    print(f"  [клієнт] отримав: {val}")
    print()

In [ ]:
# ── Повна інкапсуляція через yield у LinkedList ────────────────────────────

class EncapsulatedLinkedList:
    """
    Абсолютно інкапсульований зв'язний список.
    Клієнт не підозрює про існування _Node або _head.
    """
    
    class _Node:
        """Приватний внутрішній клас — невидимий для клієнта."""
        __slots__ = ('data', 'next')  # оптимізація пам'яті
        
        def __init__(self, data, next_node=None):
            self.data = data
            self.next = next_node
    
    def __init__(self):
        self._head = None
        self._size = 0
    
    def append(self, data) -> None:
        new_node = self._Node(data, next_node=self._head)
        self._head = new_node
        self._size += 1
    
    def __len__(self) -> int:
        return self._size
    
    def __iter__(self):
        """
        Iterator Protocol через generator.
        Обходить вузли внутрішньо, видає ТІЛЬКИ .data клієнту.
        """
        current = self._head
        while current is not None:
            yield current.data      # клієнт бачить лише дані
            current = current.next  # переходимо до наступного (секрет всередині)
    
    def __repr__(self) -> str:
        return f"[{' → '.join(str(x) for x in self)}]"


# ── Демонстрація ────────────────────────────────────────────────────────────
my_list = EncapsulatedLinkedList()
for word in ["Architecture", "Python", "Encapsulation"]:
    my_list.append(word)

print("Клієнтський код — НУЛЬОВЕ знання про вузли:")
print()

# for-loop → викликає __iter__() → отримує генератор → викликає next() у циклі
for word in my_list:
    print(f"  {word}")

print()
print(f"len(my_list) = {len(my_list)}")
print(f"repr         = {my_list}")

# Перевірка інкапсуляції: _Node — внутрішній клас
try:
    # Клієнт намагається залізти всередину
    node = my_list._head
    print(f"_head доступний (конвенція, не заборона): {node}")
except AttributeError as e:
    print(f"Помилка: {e}")

---

## Частина 6: Чотири Фундаментальні Операції

---

### 1. Обхід (Traversal) — Передача Естафети

У масиві можна "телепортуватися" до будь-якого елемента за індексом.
У вузловій структурі телепортація неможлива — ідемо від `head` по вказівниках.

Архітектурно обхід **інкапсулюється в ітераторі** — клієнт не знає механіки.

### 2. Вставка (Insertion) — Перехоплення Рук

```
До вставки:   A → B → C → None
Вставляємо X між A і B:

1. Створюємо X
2. X.next = B       (X вказує туди, куди вказував A)
3. A.next = X       (A тепер вказує на X)

Після:        A → X → B → C → None
```

Жоден елемент фізично не переміщується! **O(1)** — якщо є вказівник на попередній вузол.

### 3. Видалення (Deletion) — Обхідний Шлях

```
Видаляємо B із A → B → C:

before.next = after   (A.next = C)
B залишається "островом" в пам'яті → Garbage Collector прибере

Після: A → C → None
```

### 4. Побудова (Construction) — Динамічне Зростання

Масиви потребують заздалегідь відомого розміру → over-allocation → копіювання.
Зв'язний список: кожен вузол — незалежний маленький об'єкт. Росте нескінченно,
використовуючи розкидані фрагменти RAM без жодного копіювання.

In [ ]:
# ── Демонстрація всіх чотирьох операцій ────────────────────────────────────

class DemoLinkedList:
    """Прозора реалізація для навчальних цілей — з детальними логами."""
    
    def __init__(self):
        self._head = None
        self._size = 0
    
    def _to_list(self) -> list:
        result, cur = [], self._head
        while cur: result.append(cur.data); cur = cur.next
        return result
    
    def insert_after(self, target_data, new_data) -> bool:
        """
        Вставляє new_data ПІСЛЯ першого вузла з target_data.
        O(n) для пошуку + O(1) для вставки.
        """
        current = self._head
        while current:
            if current.data == target_data:
                # Перехоплення рук: змінюємо лише 2 вказівники
                new_node = ListNode(new_data, next_node=current.next)
                current.next = new_node
                self._size += 1
                print(f"  Вставка '{new_data}' після '{target_data}': {self._to_list()}")
                return True
            current = current.next
        return False
    
    def delete(self, target_data) -> bool:
        """Видаляє перший вузол з target_data — обхідний шлях."""
        if not self._head: return False
        
        if self._head.data == target_data:
            self._head = self._head.next   # особливий випадок: голова
            self._size -= 1
            print(f"  Видалення '{target_data}' (голова): {self._to_list()}")
            return True
        
        prev = self._head
        while prev.next:
            if prev.next.data == target_data:
                prev.next = prev.next.next  # обхідний шлях
                self._size -= 1
                print(f"  Видалення '{target_data}': {self._to_list()}")
                return True
            prev = prev.next
        return False
    
    def build_from(self, items: list) -> 'DemoLinkedList':
        """Динамічна побудова — по одному вузлу, без over-allocation."""
        for item in items:
            if self._head is None:
                self._head = ListNode(item)
            else:
                cur = self._head
                while cur.next: cur = cur.next
                cur.next = ListNode(item)
            self._size += 1
        return self


# ── Демо ─────────────────────────────────────────────────────────────────────
demo = DemoLinkedList()

print("=== 4. ПОБУДОВА (Construction) ===")
demo.build_from(["A", "B", "D", "E"])
print(f"  Після build_from(['A','B','D','E']): {demo._to_list()}")

print()
print("=== 2. ВСТАВКА (Insertion) ===")
demo.insert_after("B", "C")   # вставка між B і D

print()
print("=== 1. ОБХІД (Traversal) ===")
print("  Обхід через __iter__-стиль:")
current = demo._head
while current:
    print(f"  → {current.data}", end="")
    current = current.next
print(" → None")

print()
print("=== 3. ВИДАЛЕННЯ (Deletion) ===")
demo.delete("C")
demo.delete("A")   # видалення голови

---

## Частина 7: Стек та Черга на Вузлах

---

### Архітектурний інсайт: чому Node-based?

`collections.deque` (реалізований на C) — найкращий вибір у продакшні.
Але **розуміння Node-based реалізації** — фундамент для:
- Реалізації структур без стандартної бібліотеки
- Розуміння внутрішньої механіки `deque`
- Вирішення задач на LeetCode / технічних інтерв'ю

### Node-based Stack (LIFO)

```
push(X):   [X] → [top] → ... → None   (X стає новим top)
pop():     повертаємо top.data, top = top.next
```

In [ ]:
class Node:
    """Базовий вузол для стека і черги."""
    __slots__ = ('data', 'next')
    def __init__(self, data, next_node=None):
        self.data = data
        self.next = next_node


class LinkedStack:
    """
    Стек (LIFO) на основі зв'язного списку.
    Всі операції push/pop — строго O(1).
    """
    
    def __init__(self):
        self._top = None    # вказівник на вершину стека
        self._size = 0
    
    def push(self, data) -> None:
        """O(1): новий вузол → вказує на top → стає top."""
        self._top = Node(data, next_node=self._top)
        self._size += 1
    
    def pop(self):
        """O(1): знімаємо top, зміщуємо вказівник."""
        if self.is_empty():
            raise IndexError("pop from empty stack")
        data = self._top.data
        self._top = self._top.next   # старий top ізольований → GC приберe
        self._size -= 1
        return data
    
    def peek(self):
        """O(1): підглядаємо вершину без видалення."""
        if self.is_empty():
            raise IndexError("peek from empty stack")
        return self._top.data
    
    def is_empty(self) -> bool: return self._top is None
    def __len__(self) -> int:   return self._size
    def __repr__(self) -> str:
        parts, cur = [], self._top
        while cur: parts.append(str(cur.data)); cur = cur.next
        return f"Stack(top→[{', '.join(parts)}]→bottom)"


# ── Демонстрація ────────────────────────────────────────────────────────────
stack = LinkedStack()
print("=== LinkedStack (LIFO) ===")
for task in ["Задача A", "Задача B", "ТЕРМІНОВА C"]:
    stack.push(task)
    print(f"  push('{task}') → peek='{stack.peek()}' | size={len(stack)}")

print()
print("Обробка (LIFO):")
while not stack.is_empty():
    print(f"  pop → '{stack.pop()}'")

### Node-based Queue (FIFO)

```
enqueue(X):  head → ... → [tail] → [X] → None   (X стає новим tail)
dequeue():   повертаємо head.data, head = head.next
```

Потрібно відстежувати **обидва кінці** для O(1) на обох операціях.

In [ ]:
class LinkedQueue:
    """
    Черга (FIFO) на основі зв'язного списку.
    enqueue → O(1): додаємо до tail
    dequeue → O(1): знімаємо з head
    """
    
    def __init__(self):
        self._head = None   # звідси видаляємо (dequeue)
        self._tail = None   # сюди додаємо (enqueue)
        self._size = 0
    
    def enqueue(self, data) -> None:
        """O(1): новий вузол → чіпляємо до tail → tail = новий вузол."""
        new_node = Node(data)
        if self._tail is None:          # порожня черга
            self._head = new_node
            self._tail = new_node
        else:
            self._tail.next = new_node  # старий tail вказує на новачка
            self._tail = new_node       # tail = новачок
        self._size += 1
    
    def dequeue(self):
        """O(1): знімаємо head, зміщуємо вказівник."""
        if self.is_empty():
            raise IndexError("dequeue from empty queue")
        data = self._head.data
        self._head = self._head.next
        if self._head is None:          # черга стала порожньою
            self._tail = None           # tail теж має бути None
        self._size -= 1
        return data
    
    def peek(self):
        if self.is_empty(): raise IndexError("peek from empty queue")
        return self._head.data
    
    def is_empty(self) -> bool: return self._head is None
    def __len__(self) -> int:   return self._size
    def __repr__(self) -> str:
        parts, cur = [], self._head
        while cur: parts.append(str(cur.data)); cur = cur.next
        return f"Queue(head→[{' → '.join(parts)}]←tail)"


# ── Порівняння: list.pop(0) vs LinkedQueue.dequeue() ──────────────────────
import timeit

N = 50_000

t_list = timeit.timeit(
    stmt='q.pop(0)',
    setup=f'q = list(range({N}))',
    number=N // 2
)

# LinkedQueue
q = LinkedQueue()
for i in range(N): q.enqueue(i)

t_node = timeit.timeit(
    stmt='q.dequeue()',
    setup='',  # q вже визначений
    number=N // 2,
    globals={'q': q}
)

print(f"list.pop(0)          : {t_list:.4f} сек  — O(n)")
print(f"LinkedQueue.dequeue(): {t_node:.6f} сек — O(1)")
print(f"Прискорення          : {t_list / t_node:.1f}x")

print()

# Функціональна демонстрація
q2 = LinkedQueue()
print("=== LinkedQueue (FIFO) ===")
for req in ["Req-001", "Req-002", "Req-003"]:
    q2.enqueue(req)
    print(f"  enqueue('{req}') → {q2}")

print()
while not q2.is_empty():
    print(f"  dequeue → '{q2.dequeue()}'")

---

## Частина 8: Двонаправлений Зв'язний Список та `collections.deque`

---

### Doubly Linked List — двосторонній зв'язок

Однонаправлений список — вулиця з одностороннім рухом.
Двонаправлений — двостороннє шосе: кожен вузол має `prev` і `next`.

```
None ← [A] ⟷ [B] ⟷ [C] → None
              ↑             ↑
             head          tail
```

**Переваги:**
- Видалення вузла — O(1) **без** пошуку попереднього (бо є `prev`)
- Обхід у обох напрямках

### Python's Secret Weapon: `collections.deque`

`deque` — це **готова продакшн-реалізація** двонаправленого зв'язного списку на C.
Суворо O(1) для append/pop з **обох** кінців.

In [ ]:
class DoublyNode:
    """Вузол двонаправленого зв'язного списку."""
    __slots__ = ('data', 'prev', 'next')
    
    def __init__(self, data):
        self.data = data
        self.prev = None   # ← назад
        self.next = None   # → вперед


class DoublyLinkedList:
    """Двонаправлений зв'язний список — O(1) вставка/видалення з обох кінців."""
    
    def __init__(self):
        # Sentinel-вузли (dummy): спрощують граничні випадки
        self._sentinel_head = DoublyNode(None)  # завжди перший (не є даними)
        self._sentinel_tail = DoublyNode(None)  # завжди останній (не є даними)
        self._sentinel_head.next = self._sentinel_tail
        self._sentinel_tail.prev = self._sentinel_head
        self._size = 0
    
    def _insert_between(self, data, before: DoublyNode, after: DoublyNode):
        """O(1): вставка між двома вузлами."""
        node = DoublyNode(data)
        node.prev = before
        node.next = after
        before.next = node
        after.prev = node
        self._size += 1
        return node
    
    def _delete_node(self, node: DoublyNode):
        """O(1): видалення вузла — знаємо prev і next, тому без пошуку."""
        node.prev.next = node.next
        node.next.prev = node.prev
        self._size -= 1
        return node.data
    
    def append_right(self, data):
        """O(1): вставка перед sentinel_tail (= в кінець)."""
        return self._insert_between(data, self._sentinel_tail.prev, self._sentinel_tail)
    
    def append_left(self, data):
        """O(1): вставка після sentinel_head (= на початок)."""
        return self._insert_between(data, self._sentinel_head, self._sentinel_head.next)
    
    def pop_right(self):
        """O(1): видалення з кінця."""
        if self._size == 0: raise IndexError("pop from empty")
        return self._delete_node(self._sentinel_tail.prev)
    
    def pop_left(self):
        """O(1): видалення з початку."""
        if self._size == 0: raise IndexError("pop from empty")
        return self._delete_node(self._sentinel_head.next)
    
    def __len__(self) -> int: return self._size
    def __iter__(self):
        cur = self._sentinel_head.next
        while cur is not self._sentinel_tail:
            yield cur.data; cur = cur.next
    def __repr__(self) -> str:
        return "⟷".join(str(x) for x in self)


# ── Демонстрація ─────────────────────────────────────────────────────────────
dll = DoublyLinkedList()
dll.append_right(2)
dll.append_right(3)
dll.append_left(1)
dll.append_left(0)
print(f"Після 4 вставок: {dll}  | len={len(dll)}")

print(f"pop_right() → {dll.pop_right()} | {dll}")
print(f"pop_left()  → {dll.pop_left()}  | {dll}")

print()
print("Порівняння з collections.deque:")
from collections import deque
d = deque([1, 2, 3])
d.appendleft(0)
d.append(4)
print(f"  deque: {d}")
print(f"  d.pop()      = {d.pop()}")
print(f"  d.popleft()  = {d.popleft()}")
print(f"  Після: {d}")

---

## Частина 9: Паттерни ООП у Структурах Даних

---

### Паттерн Адаптер (Wrapper) — обмеження інтерфейсу

Ви маєте `list` з десятками методів, але хочете надати клієнту лише LIFO-інтерфейс.
Адаптер обгортає список і **приховує** непотрібні методи.

```
Клієнт → stack.push(x)  → _data.append(x)   ← list
Клієнт → stack.pop()    → _data.pop()        ← list
Клієнт → ??? insert(0, x) ???  НЕДОСТУПНО    ← прихований адаптером
```

In [ ]:
# ── Паттерн Адаптер: Stack через list ────────────────────────────────────────

class Stack:
    """
    Адаптер: обгортає list, надає лише LIFO-інтерфейс.
    Клієнт не може зробити insert(0, x) — це LIFO-контракт.
    """
    
    def __init__(self): self._data: list = []
    
    def push(self, item) -> None: self._data.append(item)
    def pop(self): 
        if not self._data: raise IndexError("pop from empty stack")
        return self._data.pop()
    def peek(self): return self._data[-1] if self._data else None
    def is_empty(self) -> bool: return not self._data
    def __len__(self): return len(self._data)
    def __repr__(self): return f"Stack({self._data})"


# ── Паттерн Відвідувач (Visitor) — алгоритм окремо від структури ─────────────

class TreeNode:
    """Вузол дерева — зберігає дані і дочірні елементи."""
    def __init__(self, value, children=None):
        self.value = value
        self.children = children or []
    
    def __iter__(self):
        """DFS-обхід через yield from — ітератор, що не розкриває структуру."""
        yield self.value
        for child in self.children:
            yield from child   # рекурсивна делегація


class NodeVisitor:
    """
    Паттерн Відвідувач: алгоритм відокремлений від структури.
    Node-класи відповідають лише за зберігання + зв'язки.
    Вся бізнес-логіка — у Visitor.
    """
    
    def visit(self, node):
        """Динамічна диспетчеризація: викликає visit_ClassName."""
        method_name = f'visit_{type(node).__name__}'
        method = getattr(self, method_name, self.generic_visit)
        return method(node)
    
    def generic_visit(self, node):
        return f"<{type(node).__name__}>"


class SizeVisitor(NodeVisitor):
    """Відвідувач, що підраховує кількість вузлів."""
    
    def visit_TreeNode(self, node: TreeNode) -> int:
        return 1 + sum(self.visit(child) for child in node.children)


class PrintVisitor(NodeVisitor):
    """Відвідувач, що красиво друкує дерево."""
    
    def visit_TreeNode(self, node: TreeNode, depth: int = 0) -> None:
        print("  " * depth + f"├─ {node.value}")
        for child in node.children:
            self.visit_TreeNode(child, depth + 1)


# ── Демонстрація ─────────────────────────────────────────────────────────────
tree = TreeNode("root", [
    TreeNode("A", [TreeNode("A1"), TreeNode("A2")]),
    TreeNode("B", [TreeNode("B1")]),
    TreeNode("C"),
])

print("=== Адаптер: Stack ===")
s = Stack()
for x in [1, 2, 3]: s.push(x)
print(f"  {s} → pop={s.pop()}")

print()
print("=== TreeNode.__iter__ (DFS через yield from) ===")
print("  Порядок обходу:", list(tree))

print()
print("=== Відвідувач: SizeVisitor ===")
size_visitor = SizeVisitor()
print(f"  Кількість вузлів: {size_visitor.visit(tree)}")

print()
print("=== Відвідувач: PrintVisitor ===")
print_visitor = PrintVisitor()
print_visitor.visit(tree)

---

## Частина 10: Головний Інсайт — Структура Даних = Організація Пам'яті

---

> Структура даних — це не абстрактний набір методів.
> Це **фундаментальне рішення про те, як ваші дані фізично ляжуть на мікросхеми RAM**.

| Вибір | Фізична вимога | Виграш | Жертва |
|-------|---------------|--------|--------|
| **Масив** (list) | Монолітний блок RAM | O(1) довільний доступ, CPU cache locality | O(n) вставка/видалення, over-allocation |
| **Вузлові структури** | Децентралізовані острівці | O(1) вставка/видалення, динамічне зростання | O(n) доступ за індексом, overhead на вказівники |

### CPU Cache Locality — прихований бонус масивів

Сучасні CPU завантажують дані блоками (cache lines, ~64 байти).
Якщо ваш масив поміщається в кеш — ітерація у **10-100x швидша**,
ніж ітерація зв'язного списку (вузли розкидані по RAM → cache miss на кожному кроці).

```
Масив: [10][20][30][40][50] ← все в одному cache line → блискавично
Linked: [10]→...[20]→...[30] ← кожен стрибок = cache miss → повільно
```

> **Senior-insight:** Для читання + довільного доступу — масив швидший навіть при O(n) доступі,
> бо CPU cache компенсує. Зв'язний список ефективний лише там, де **дійсно** потрібні O(1) вставки.

---

## Питання для Співбесіди

---

### 1. Чим Python `list` відрізняється від зв'язного списку архітектурно?

> **Відповідь:**
> Python `list` — це динамічний масив: суцільний блок RAM із масивом вказівників.
> Зв'язний список — децентралізовані вузли, розкидані по RAM, з'єднані вказівниками `next`.
> `list` дає O(1) доступ за індексом, але O(n) для insert/delete на початку.
> Linked list дає O(1) insert/delete (якщо є вказівник), але O(n) пошук за індексом.

---

### 2. Навіщо ховати `_Node` і `_head` від клієнта?

> **Відповідь:**
> Це не безпека — це **фізична незалежність даних**.
> Якщо клієнт залежить від `node.next`, ми не можемо змінити реалізацію
> на масив без зламу клієнтського коду. Ховаючи деталі за інтерфейсом
> (ADT), ми зберігаємо свободу повністю переписати внутрішнє без
> жодних наслідків для клієнтів.

---

### 3. Поясніть механізм `yield` у `__iter__`. Як Python будує ітератор?

> **Відповідь:**
> `yield` перетворює функцію на генераторну — при виклику повертає
> генераторний об'єкт (ітератор). Кожен `next()` відновлює виконання
> з останньої точки `yield`, зберігаючи стан локальних змінних.
> `for x in obj:` неявно викликає `iter(obj)` → `__iter__()` → генератор,
> потім `next()` у циклі до `StopIteration`.

---

### 4. Яка складність вставки у середину зв'язного списку?

> **Відповідь:**
> Якщо є вказівник на вузол-попередник: **O(1)** — змінюємо два вказівники.
> Якщо треба спочатку знайти позицію: **O(n)** для пошуку + O(1) для вставки = O(n) загалом.
> Саме тому зв'язні списки вигідні для вставок **тільки** коли є прямий доступ до вузла.

---

### 5. Що таке sentinel-вузол і навіщо він потрібен?

> **Відповідь:**
> Sentinel (dummy) — фіктивний вузол на початку (і/або кінці) списку,
> що не містить реальних даних. Він усуває граничні випадки:
> вставка в порожній список, видалення голови — завжди однаковий код,
> без `if self._head is None`. Спрощує реалізацію і зменшує кількість помилок.

---

### 6. Чому `collections.deque` не замінює зв'язний список повністю?

> **Відповідь:**
> `deque` ефективний лише на **обох кінцях** — O(1) для append/pop.
> Доступ до середини: `deque[i]` — O(n). Вставка в середину — O(n).
> Зв'язний список (якщо є вказівник на вузол) дає O(1) вставку/видалення
> **будь-де**. `deque` — оптимізований спецслучай для двосторонніх операцій.

---

### 7. Поясніть паттерн Відвідувач стосовно структур даних.

> **Відповідь:**
> Паттерн Відвідувач відокремлює алгоритм від структури.
> Node-класи відповідають лише за зберігання даних і підтримку зв'язків.
> Вся бізнес-логіка (обчислення, серіалізація, типізація) — у Visitor.
> Це дотримання SRP: структура — стабільна, Visitor — змінний.

---

---

## Домашні Завдання

---

### Завдання 1 — Реверс Зв'язного Списку (Рівень: Легкий)

Реалізуйте функцію `reverse_linked_list(head)`, яка реверсує однонаправлений
зв'язний список **in-place** (без додаткової пам'яті, крім O(1) змінних).

**Вхід:** `1 → 2 → 3 → 4 → 5 → None`
**Вихід:** `5 → 4 → 3 → 2 → 1 → None`

**Підказка:** три вказівники: `prev`, `current`, `next_node`.

In [ ]:
# === Завдання 1: Реверс зв'язного списку ===

class ListNode:
    def __init__(self, data, next_node=None):
        self.data = data
        self.next = next_node
    def __repr__(self): return f"Node({self.data})"


def build_list(values: list) -> ListNode | None:
    """Допоміжна функція: будує зв'язний список із списку значень."""
    if not values: return None
    head = ListNode(values[0])
    cur = head
    for v in values[1:]: cur.next = ListNode(v); cur = cur.next
    return head


def to_python_list(head: ListNode | None) -> list:
    """Конвертує зв'язний список у Python list для перевірки."""
    result, cur = [], head
    while cur: result.append(cur.data); cur = cur.next
    return result


def reverse_linked_list(head: ListNode | None) -> ListNode | None:
    """
    TODO: Реалізуйте реверс in-place.
    Складність: O(n) час, O(1) пам'ять.
    """
    pass


# Тести
head = build_list([1, 2, 3, 4, 5])
print(f"До реверсу  : {to_python_list(head)}")
reversed_head = reverse_linked_list(head)
print(f"Після реверсу: {to_python_list(reversed_head)}")
print(f"Правильно   : {to_python_list(reversed_head) == [5, 4, 3, 2, 1]}")

### Завдання 2 — Черга через два Стеки (Рівень: Середній)

Реалізуйте клас `QueueFromTwoStacks`, що емулює FIFO-чергу,
використовуючи **два стеки** (два списки Python).

**API:** `enqueue(x)`, `dequeue()`, `is_empty()`

**Складність:** `enqueue` — O(1), `dequeue` — амортизовано O(1).

**Підказка:** `stack_in` — для вхідних елементів;
при `dequeue` — перекладаємо всі у `stack_out` (один раз), потім pop.

In [ ]:
# === Завдання 2: Черга через два стеки ===

class QueueFromTwoStacks:
    def __init__(self):
        self._stack_in: list = []    # для enqueue
        self._stack_out: list = []   # для dequeue
    
    def enqueue(self, data) -> None:
        """TODO: O(1) — кладемо в stack_in."""
        pass
    
    def dequeue(self):
        """
        TODO: амортизоване O(1).
        Якщо stack_out порожній — перекидаємо всі з stack_in.
        Потім pop зі stack_out.
        """
        pass
    
    def is_empty(self) -> bool:
        return not self._stack_in and not self._stack_out


# Тести
q = QueueFromTwoStacks()
for x in [1, 2, 3, 4, 5]:
    q.enqueue(x)

results = []
while not q.is_empty():
    results.append(q.dequeue())

print(f"Результат dequeue: {results}")
print(f"Правильно (FIFO):  {results == [1, 2, 3, 4, 5]}")

### Завдання 3 — LRU Cache (Рівень: Складний)

Реалізуйте клас `LRUCache` (Least Recently Used Cache) з операціями:
- `get(key)` → повертає значення або -1 якщо немає
- `put(key, value)` → додає/оновлює; якщо cache повний — видаляє найстарший елемент

**Складність:** обидві операції — O(1).

**Архітектурна підказка:** використайте `dict` (для O(1) пошуку) +
`collections.OrderedDict` або власний двонаправлений зв'язний список
(для відстеження порядку використання).

In [ ]:
from collections import OrderedDict

# === Завдання 3: LRU Cache ===

class LRUCache:
    """
    Least Recently Used Cache.
    Реалізується через OrderedDict: move_to_end() → O(1).
    """
    
    def __init__(self, capacity: int):
        self.capacity = capacity
        # TODO: ініціалізуйте OrderedDict
        pass
    
    def get(self, key: int) -> int:
        """
        TODO:
        - якщо key є → переміщуємо в кінець (MRU) → повертаємо значення
        - якщо немає → повертаємо -1
        """
        pass
    
    def put(self, key: int, value: int) -> None:
        """
        TODO:
        - якщо key вже є → оновлюємо + переміщуємо в кінець
        - якщо cache повний → видаляємо найстаріший (перший у OrderedDict)
        - додаємо новий елемент в кінець
        """
        pass


# Тести
cache = LRUCache(capacity=3)
cache.put(1, 10)
cache.put(2, 20)
cache.put(3, 30)
print(f"get(1) = {cache.get(1)}")   # 10 — переміщує 1 в кінець
cache.put(4, 40)                    # видаляє 2 (LRU), додає 4
print(f"get(2) = {cache.get(2)}")   # -1 — 2 було видалено
print(f"get(3) = {cache.get(3)}")   # 30
print(f"get(4) = {cache.get(4)}")   # 40

---

## Підсумок Уроку

---

| Концепція | Ключова думка |
|-----------|---------------|
| **Python `list`** | Динамічний масив: O(1) доступ, O(n) insert/delete |
| **Linked List** | Вузли-острівці: O(1) insert/delete (при вказівнику), O(n) доступ |
| **ADT** | Контракт (що) відокремлений від реалізації (як) |
| **Інкапсуляція** | Ховаємо внутрішнє — звільняємо себе від зобов'язань |
| **Iterator Pattern** | `__iter__` + `yield`: клієнт обходить, не знаючи структури |
| **Sentinel Node** | Фіктивний вузол усуває граничні випадки |
| **Adapter Pattern** | Обгортаємо `list` → отримуємо строгий LIFO/FIFO |
| **Visitor Pattern** | Алгоритм → окремий клас; Node — лише дані + зв'язки |
| **collections.deque** | C-реалізація doubly linked list: O(1) з обох кінців |

> **Головний інсайт:**
> Структура даних — це рішення про те, **як дані фізично ляжуть на мікросхеми RAM**.
> Розуміння цього дозволяє передбачати вузькі місця ще до написання першого рядка коду.

---
*Наступний урок: Алгоритми пошуку та хешування*